# Preliminary examples

In [ ]:
#    APM41012EP course notebook - Chapter 4 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Preliminary examples
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np
import sympy as sp
from mpmath import mp
import mpmath

## Hilbert matrix

A Hilbert matrix is a square matrix with general term:

$$ H_{ij} = \frac{1}{i+j-1} $$

The Hilbert matrix of size 6 is written:

$$\begin{pmatrix}
 1   & 1/2 & 1/3 & 1/4 & 1/5  & 1/6  \\
 1/2 & 1/3 & 1/4 & 1/5 & 1/6  & 1/7  \\
 1/3 & 1/4 & 1/5 & 1/6 & 1/7  & 1/8  \\
 1/4 & 1/5 & 1/6 & 1/7 & 1/8  & 1/9  \\
 1/5 & 1/6 & 1/7 & 1/8 & 1/9  & 1/10 \\
 1/6 & 1/7 & 1/8 & 1/9 & 1/10 & 1/11  
\end{pmatrix}$$

## Arbitrary-precision solution

First, we compute, with SymPy, the exact solution for a right-hand side $b$ whose components are all equal to 1, for $n = 6$:

In [ ]:
n = 6
A = sp.Matrix(n, n, lambda i, j: sp.Rational(1, i+j+1))
print("A = ")
sp.pprint(A, wrap_line=False)

b = sp.ones(n, 1)
print("\nb = ")
sp.pprint(b, wrap_line=False)

x = A.solve(b)
print("\nSolution of Ax=b: ")
sp.pprint(x, wrap_line=False)

Then, we solve the same system for different floating-point number formats with the mpmath library:

In [ ]:
print("Exact solution  :")
sp.pprint(x, wrap_line=False)

def hilbert_mp(n):   # the entries are rounded at the current mpmath precision
    return mpmath.matrix([[mp.mpf(1)/(i+j-1) for j in range(1, n+1)] for i in range(1, n+1)])

# double precision
mp.prec = 53
A = hilbert_mp(n)
b = mpmath.matrix([1 for i in range(0,n)])
x = mpmath.lu_solve(A, b)
print(f"\nSolution with a precision of {mp.dps} significant digits (double precision): ")
print(x)

# simple precision
mp.prec = 24
A = hilbert_mp(n)
b = mpmath.matrix([1 for i in range(0,n)])
x = mpmath.lu_solve(A, b)
print(f"\nSolution with a precision of {mp.dps} significant digits (single precision): ")
print(x)

# 3 significant digits
mp.dps = 4
A = hilbert_mp(n)
b = mpmath.matrix([1 for i in range(0,n)])
x = mpmath.lu_solve(A, b)
print(f"\nSolution with a precision of {mp.dps} significant digits: ")
print(x)

## Perturbation of the right-hand side for an exact solve

We initialise a Hilbert matrix of size $n$ with rational fractions so as to perform exact computations.

In [ ]:
n = 6
A = sp.Matrix(n, n, lambda i, j: sp.Rational(1, i+j+1))
sp.pprint(A, wrap_line=False)

We compute the exact solution for a right-hand side $b$ whose components are all equal to 1

In [ ]:
b = sp.ones(n, 1)
x = A.solve(b)
sp.pprint(x, wrap_line=False)

We perturb the last component of the right-hand side by (1/1000000), we solve the system and we expect to find a solution close to the vector whose components are all $1$:

In [ ]:
b[n-1] *= (1 + sp.Rational(1, 1000000)) #  perturbation of the right-hand side
x_pert = A.solve(b)
print("Perturbed solution:")
sp.pprint(x_pert, wrap_line=False)

In [ ]:
print("Difference between the solution and the perturbed solution:" )
for i in range(n):
    print(abs(float(x[i]-x_pert[i])))

err = max(abs(float(x[i]-x_pert[i])) for i in range(0,n))
print("\nInfinity norm of the difference between the solution and the perturbed solution  :", err)

## Forsythe's example

We consider the system:

$$
\begin{pmatrix}
 0.0001   & 1 \\
 1   & 1 
\end{pmatrix}
\begin{pmatrix}
 x_1  \\
 x_2  
\end{pmatrix}
=
\begin{pmatrix}
 1  \\
 2  
\end{pmatrix}
$$

for which the exact solution is $\displaystyle x_1 = \frac{10000}{9999}$ and $\displaystyle x_2 = \frac{9998}{9999}$, that is $x_1 = 1.00010001000100\dots$ and $x_2 = 0.999899989999$

We use the Gaussian elimination algorithm with floating-point numbers having 3 significant digits (in base 10).

In [ ]:
mp.prec = 12
print(f"Floating-point precision with {mp.dps} significant digits")

Solving the system:

In [ ]:
a11 = mp.mpf('1e-4')
a12 = mp.mpf('1')
a21 = mp.mpf('1')
a22 = mp.mpf('1')

b1 = mp.mpf('1')
b2 = mp.mpf('2')

# gaussian elimination
a22 = a22 - (a21/a11)*a12
b2  = b2 - (a21/a11)*b1

# backward substitution
x2 = b2 / a22
x1 = (b1 - a12*x2) / a11

print("Solution")
print("x1 = ", x1)
print("x2 = ", x2)

We swap the 2 rows of the system: 

$$
\begin{pmatrix}
 1.       & 1 \\
  0.0001  & 1 
\end{pmatrix}
\begin{pmatrix}
 x_2  \\
 x_1  
\end{pmatrix}
=
\begin{pmatrix}
 2 \\ 
 1 
\end{pmatrix}
$$

In [ ]:
a11 = mp.mpf('1')
a12 = mp.mpf('1')
a21 = mp.mpf('1e-4')
a22 = mp.mpf('1')

b1 = mp.mpf('2')
b2 = mp.mpf('1')

# gaussian elimination
a22 = a22 - (a21/a11)*a12
b2  = b2 - (a21/a11)*b1

# backward substitution
x2 = b2 / a22
x1 = (b1 - a12*x2) / a11

print("Solution")
print("x1 = ", x1)
print("x2 = ", x2)